# 1. CrowdHumanを用いたYOLO追加学習
vbox=可視人物、hbox=頭部、both=person/headの2クラスです。モデルは参考実装のYOLO26nを使用します。
[公式配布・条件](https://www.crowdhuman.org/download.html) は非商用の研究・教育用途に限定し、画像再配布を禁止しています。
[Ultralyticsライセンス](https://www.ultralytics.com/license) も確認してください。

実データは `/content/crowdhuman/` だけに保存します。Google Driveはマウントしません。
ColabはGoogleのクラウドです。利用条件でクラウド処理を認められない場合はこのノートブックを実行せず、ローカル用に別途移植してください。
ランタイム終了・リセットでデータと重みが失われます。画像・アノテーションの表示、外部送信、出力保存は禁止します。
Colabの「編集 → ノートブックの設定 → 保存時にコードセルの出力を除外する」を有効にしてください。
このノートブック自身にも出力除外メタデータを設定しています。共有前に全出力を消してください。

GPU学習・実データ・ブラウザアプリとの接続は開発環境では未検証です。

## 2. Colab実行環境とGPUの確認
Colabの「ランタイム → ランタイムのタイプを変更」でGPUを選択します。GPUがなければ警告し、学習時に停止します。

In [ ]:
import os, sys, subprocess, warnings
from pathlib import Path
if 'google.colab' not in sys.modules:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('Google Colabで実行してください') from None
BOOTSTRAP = Path('/content/crowdhuman')
if BOOTSTRAP.resolve() != BOOTSTRAP:
    raise RuntimeError('保存先が別の場所へリンクされています')
BOOTSTRAP.mkdir(parents=True, exist_ok=True)
os.environ.update(YOLO_CONFIG_DIR=str(BOOTSTRAP / 'config'), YOLO_OFFLINE='true',
                  YOLO_AUTOINSTALL='false', YOLO_VERBOSE='false', WANDB_MODE='disabled',
                  WANDB_DISABLED='true', COMET_MODE='DISABLED', HF_HUB_OFFLINE='1')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True) if __import__('shutil').which('nvidia-smi') else None
if gpu is None or gpu.returncode:
    warnings.warn('GPUが見つかりません。GPUランタイムを選択してください。')
else:
    print(gpu.stdout.strip())

## 3. ライブラリのインストール
Ultralyticsを8.4.52に固定します。ColabのCUDA対応PyTorchを利用し、実際の依存バージョンは実験に記録します。
インストールログはランタイム内に保存します。依存を変更した場合はランタイムを再起動して最初から実行してください。

In [ ]:
REQUIREMENTS = '# ColabのCUDA対応torch/torchvisionを維持。実際の全バージョンは実行時に記録。\nultralytics==8.4.52\nPillow>=10,<13\nPyYAML>=6,<7\nonnx>=1.17,<2\nonnxruntime>=1.20,<2\n'
requirements_path = BOOTSTRAP / 'requirements.txt'
requirements_path.write_text(REQUIREMENTS)
with open(BOOTSTRAP / 'install.log', 'w') as log:
    installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], stdout=log, stderr=log)
if installed.returncode:
    raise RuntimeError('インストール失敗。ランタイム内のinstall.logを確認してください。')
print('インストール完了')

## 4. 学習設定
初回は `DEBUG_MODE=True`、少ないepochで動作確認してください。正式比較ではFalseに戻し、別のRUN_NAMEを使います。
`MODES_TO_RUN=["vbox", "hbox"]` で同じ設定の比較を一括実行できます。bothは2クラスです。
利用条件とクラウド処理の可否を確認後に `TERMS_REVIEWED=True` にします。認証情報は入力しません。
中断した同一実験だけ `RESUME=True` にします。設定・データ・依存バージョンが変わる再開は拒否します。

In [ ]:
TRAIN_MODE = 'vbox'
MODES_TO_RUN = [TRAIN_MODE]  # 比較例: ['vbox', 'hbox']; 3モード: ['vbox', 'hbox', 'both']
BASE_MODEL = 'yolo26n.pt'
EPOCHS = 50
BATCH_SIZE = 16
IMAGE_SIZE = 640
SEED = 42
DEBUG_MODE = False
DEBUG_LIMIT = 32  # 各splitの先頭N画像。すべてのモードで同じ画像。
RUN_NAME = 'experiment'
RESUME = False
ALLOW_CPU = False
TERMS_REVIEWED = False
AUTO_DOWNLOAD = True
EXPORT_ONNX = True
WEBAPP_CONTRACT = None  # 既存アプリから採取したinput/outputs辞書。未指定では互換性を確定しません。
BASE_DIR = '/content/crowdhuman'
RAW_DIR = f'{BASE_DIR}/raw'
PROCESSED_DIR = f'{BASE_DIR}/processed'
RUNS_DIR = f'{BASE_DIR}/runs'
EXPORT_DIR = f'{BASE_DIR}/exports'
assert TRAIN_MODE in ('vbox', 'hbox', 'both')
assert MODES_TO_RUN and len(set(MODES_TO_RUN)) == len(MODES_TO_RUN)
assert all(mode in ('vbox', 'hbox', 'both') for mode in MODES_TO_RUN)

## 5. ディレクトリと非公開設定
以下は通常のPython関数です。ノートブック内で直接確認・編集でき、srcファイルの取得・生成・importは不要です。
raw/archives・raw/images・raw/annotations・processed・runs・exportsを作成します。
外部追跡サービス・同期・analyticsのコールバックを無効にし、学習ログもこの配下だけに保存します。
GitHubで管理する開発用srcとこのノートブックは、tools/build_notebook.pyで同期できます。Colabでの実行にはこの.ipynbだけを使います。

In [ ]:
"""Enforce runtime storage and disable integrations before importing YOLO."""
import os
import contextlib
import logging
from pathlib import Path

BASE = Path('/content/crowdhuman')


def runtime_path(path):
    path = Path(path).absolute()
    resolved = path.resolve()
    if resolved != BASE and BASE not in resolved.parents:
        raise ValueError('CrowdHuman paths must stay under /content/crowdhuman')
    return path


def initialize(base=None):
    base = runtime_path(BASE if base is None else base)
    if base != BASE:
        raise ValueError('BASE_DIR must be /content/crowdhuman')
    for part in ('raw/archives', 'raw/images', 'raw/annotations', 'processed', 'runs', 'exports', 'weights', 'config'):
        runtime_path(base / part).mkdir(parents=True, exist_ok=True)
    os.environ.update(YOLO_CONFIG_DIR=str(base / 'config'), YOLO_OFFLINE='true',
                      YOLO_AUTOINSTALL='false', YOLO_VERBOSE='false',
                      WANDB_MODE='disabled', COMET_MODE='DISABLED',
                      WANDB_DISABLED='true', HF_HUB_OFFLINE='1')
    return base


def private_yolo():
    initialize()
    from ultralytics import YOLO, settings
    keys = ('sync', 'hub', 'wandb', 'mlflow', 'clearml', 'comet', 'dvc', 'neptune', 'raytune', 'tensorboard')
    settings.update({key: False for key in keys if key in settings})
    settings.update({'datasets_dir': str(BASE / 'processed'),
                     'weights_dir': str(BASE / 'weights'), 'runs_dir': str(BASE / 'runs')})
    # Disable integration registration, including Platform and analytics callbacks.
    from ultralytics.utils import callbacks
    callbacks.add_integration_callbacks = lambda instance: None
    from ultralytics.utils.callbacks import base
    base.add_integration_callbacks = lambda instance: None
    return YOLO


@contextlib.contextmanager
def local_console(path):
    """Capture pre-existing YOLO log handlers as well as stdout/progress bars."""
    path = runtime_path(path)
    logger = logging.getLogger('ultralytics')
    old_handlers, old_propagate = logger.handlers[:], logger.propagate
    with open(path, 'a') as log:
        logger.handlers = [logging.StreamHandler(log)]
        logger.propagate = False
        try:
            with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                yield
        finally:
            logger.handlers, logger.propagate = old_handlers, old_propagate

GITIGNORE_CONTENT = 'CrowdHuman/\ncrowdhuman/\ndatasets/\ndata/\n*.odgt\n*.zip\n*.jpg\n*.jpeg\n*.png\n*.webp\nruns/\nweights/\ncheckpoints/\nexports/\n*.pt\n*.pth\n*.onnx\n.env\n.env.*\n*.key\n__pycache__/\n.ipynb_checkpoints/\n.venv/\n.pytest_cache/\n*.cache\n'
BASE = initialize(Path(BASE_DIR))
with local_console(BASE / 'setup.log'):
    YOLO = private_yolo()
print('ランタイム内の保存先とプライバシー設定を準備しました')

## 6. CrowdHumanの公式取得またはアップロード
公式ページが現在案内する公開リンクのみを使います。配布元がGoogle Driveでも、ファイルの保存先はColabの一時領域です。
認証・確認画面・アクセス制限・通信失敗があれば自動取得を停止します。回避処理はありません。
その場合だけ、公式から正規に取得した不足ファイルをアップロードします。AUTO_DOWNLOAD=Falseは、既に自動取得不能と確認した場合の手動経路です。
大きいZIPのブラウザアップロードはRAMを消費するため、必要ならファイルを1つずつ選び、このセルを再実行してください。
正常な既存ファイルは再取得しません。破損ファイルは確認のうえランタイム内から削除・交換してください。

In [ ]:
"""Official downloads, fail-closed validation, safe extraction; no image display."""
import json
import re
import shutil
import stat
import urllib.request
import zipfile
import zlib
from pathlib import Path

OFFICIAL_PAGE = 'https://www.crowdhuman.org/download.html'
ARCHIVES = ('CrowdHuman_train01.zip', 'CrowdHuman_train02.zip', 'CrowdHuman_train03.zip', 'CrowdHuman_val.zip')
ANNOTATIONS = ('annotation_train.odgt', 'annotation_val.odgt')
EXPECTED = {'train': 15000, 'val': 4370}


def destination(name):
    if name not in ARCHIVES + ANNOTATIONS:
        raise ValueError('Unexpected dataset filename')
    return BASE / 'raw' / ('archives' if name in ARCHIVES else 'annotations') / name


def read_odgt(path):
    """Never include raw annotations or IDs in exception messages."""
    runtime_path(path)
    with open(path, encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, 1):
            try:
                record = json.loads(line)
                image_id = record['ID']
                if not isinstance(image_id, str) or not re.fullmatch(r'[A-Za-z0-9_-]+', image_id):
                    raise ValueError()
                if not isinstance(record['gtboxes'], list):
                    raise ValueError()
            except (ValueError, TypeError, KeyError):
                raise ValueError(f'Invalid ODGT record at line {line_no}') from None
            yield record


def validate_file(path, name):
    runtime_path(path)
    if not Path(path).is_file():
        raise FileNotFoundError(f'Missing required file: {name}')
    if name in ARCHIVES:
        try:
            with zipfile.ZipFile(path) as z:
                if z.testzip() is not None:
                    raise ValueError('ZIP CRC check failed')
                if not z.infolist():
                    raise ValueError('Empty ZIP')
        except zipfile.BadZipFile:
            raise ValueError(f'Invalid ZIP: {name}') from None
        return None
    count = sum(1 for _ in read_odgt(path))
    if count == 0:
        raise ValueError('Empty annotation file')
    return count


def official_links():
    """Resolve only public Google links currently advertised by the official page."""
    with urllib.request.urlopen(OFFICIAL_PAGE, timeout=60) as response:
        page = response.read().decode('utf-8')
    found = {}
    for name in ARCHIVES + ANNOTATIONS:
        match = re.search(re.escape(name) + r'(?:(?!CrowdHuman_\w+\.zip|annotation_\w+\.odgt).)*?href=[\"\'](https://drive\.google\.com/[^\"\']+)', page, re.S)
        if match:
            drive_id = re.search(r'/d/([A-Za-z0-9_-]+)', match[1])
            if drive_id:
                found[name] = f'https://drive.google.com/uc?export=download&id={drive_id[1]}'
    return found


def download_official():
    """No cookies, authentication, quota bypass or confirmation-token workaround."""
    needed = []
    for name in ARCHIVES + ANNOTATIONS:
        path = destination(name)
        if path.exists():
            validate_file(path, name)  # Corrupt files require explicit removal/replacement.
        else:
            needed.append(name)
    if not needed:
        return
    links = official_links()
    for name in needed:
        if name not in links:
            raise RuntimeError('Official public link unavailable; use lawful manual upload')
        path = destination(name)
        runtime_path(path)
        temp = path.with_suffix(path.suffix + '.part')
        try:
            with urllib.request.urlopen(links[name], timeout=120) as response, open(temp, 'wb') as out:
                if 'text/html' in response.headers.get('Content-Type', ''):
                    raise RuntimeError('Public download unavailable; use lawful manual upload')
                length = int(response.headers.get('Content-Length', 0))
                if length and shutil.disk_usage(BASE).free < length + 2 * 1024**3:
                    raise OSError('Insufficient download disk space')
                while chunk := response.read(1024**2):
                    if shutil.disk_usage(BASE).free < len(chunk) + 1024**3:
                        raise OSError('Insufficient download disk space')
                    out.write(chunk)
            validate_file(temp, name)
            temp.replace(path)
        except Exception:
            temp.unlink(missing_ok=True)
            raise RuntimeError(f'Download failed: {name}; use lawful manual upload') from None


def upload_missing():
    """Colab upload widget writes files directly to raw; it never prints file contents."""
    from google.colab import files
    import os
    previous = Path.cwd()
    try:
        os.chdir(runtime_path(BASE / 'raw'))
        uploaded = files.upload()
        del uploaded  # Browser upload may use substantial RAM for large archives.
        for name in ARCHIVES + ANNOTATIONS:
            staged = BASE / 'raw' / name
            if staged.exists():
                target = destination(name)
                if target.exists():
                    validate_file(target, name)
                    staged.unlink()
                else:
                    validate_file(staged, name)
                    staged.replace(target)
    finally:
        os.chdir(previous)


def inventory():
    counts = {}
    for name in ARCHIVES + ANNOTATIONS:
        count = validate_file(destination(name), name)
        if count is not None:
            counts[name] = count
    for split in EXPECTED:
        if counts[f'annotation_{split}.odgt'] != EXPECTED[split]:
            raise ValueError('Official annotation count mismatch')
    return {'annotation_counts': counts, 'free_gib': round(shutil.disk_usage(BASE).free / 1024**3, 2)}


def extract_archives():
    root = runtime_path(BASE / 'raw/images')
    entries = []
    seen = set()
    for name in ARCHIVES:
        archive = destination(name)
        validate_file(archive, name)
        with zipfile.ZipFile(archive) as z:
            for item in z.infolist():
                if item.is_dir():
                    continue
                parts = Path(item.filename).parts
                if len(parts) != 2 or parts[0] != 'Images' or Path(parts[1]).suffix.lower() not in ('.jpg', '.jpeg'):
                    raise ValueError('Unexpected archive member')
                if stat.S_ISLNK(item.external_attr >> 16) or parts[1] in seen:
                    raise ValueError('Duplicate/unsafe archive member')
                seen.add(parts[1])
                target = runtime_path(root / parts[1])
                entries.append((archive, item.filename, target, item.file_size, item.CRC))
    required = sum(size for _, _, p, size, _ in entries if not p.exists() or p.stat().st_size != size)
    if shutil.disk_usage(BASE).free < required + 2 * 1024**3:
        raise OSError('Insufficient extraction space (plus 2 GiB reserve)')
    # Compare CRCs before reuse; a same-size corrupt image must also be replaced.
    for archive in map(destination, ARCHIVES):
        with zipfile.ZipFile(archive) as z:
            for source_archive, member, target, size, crc in entries:
                if source_archive != archive:
                    continue
                if target.exists() and target.stat().st_size == size:
                    actual = 0
                    with open(target, 'rb') as existing:
                        for chunk in iter(lambda: existing.read(1024**2), b''):
                            actual = zlib.crc32(chunk, actual)
                    if actual == crc:
                        continue
                part = target.with_suffix('.part')
                with z.open(member) as source, open(part, 'wb') as out:
                    shutil.copyfileobj(source, out)
                part.replace(target)
    return {'archive_images': len(entries)}


def image_size(path):
    runtime_path(path)
    from PIL import Image
    try:
        with Image.open(path) as image:
            size = image.size
            image.verify()
        if min(size) <= 0:
            raise ValueError()
        return size
    except Exception:
        raise ValueError('Missing or corrupt dataset image') from None


def inspect_dataset():
    """Check every image, including DEBUG runs; reject split overlap and duplicates."""
    splits = {}
    sizes = {}
    for split in ('train', 'val'):
        ids = set()
        for record in read_odgt(destination(f'annotation_{split}.odgt')):
            key = record['ID']
            if key in ids:
                raise ValueError('Duplicate image ID within split')
            ids.add(key)
            sizes[key] = image_size(BASE / 'raw/images' / f'{key}.jpg')
        splits[split] = ids
    if splits['train'] & splits['val']:
        raise ValueError('Train/val overlap detected')
    return sizes, {split: len(ids) for split, ids in splits.items()}

if not TERMS_REVIEWED:
    raise RuntimeError('公式利用条件とColabでの処理可否を確認し、TERMS_REVIEWED=Trueにしてください。')
if AUTO_DOWNLOAD:
    try:
        download_official()
        print('必要ファイルを取得・再利用しました')
    except Exception:
        print('自動取得できません。公式から正規取得した不足ファイルを選択してください。')
        upload_missing()
else:
    upload_missing()

## 7. ファイル整合性・空き容量確認
必須6ファイル、全ZIPのCRC、ODGT構造と公式の件数(train 15,000 / val 4,370)を確認します。
データの内容や画像IDは表示せず件数だけを出力します。展開前に非圧縮サイズと空き容量も確認します。

In [ ]:
print(inventory())

## 8. ZIPの安全な解凍
raw/imagesに1組だけ展開します。想定外のパス・シンボリックリンク・重複画像を拒否します。
既存画像はCRCが一致すれば再展開しません。画像・アノテーションはすべてランタイム内に留まります。

In [ ]:
print(extract_archives())

## 9. ODGTの解析と画像の検証
1行ずつJSONとして読み込み、全画像の存在・画像ヘッダの整合性・サイズを調べます。
同一split内の重複ID、train/val間のID重複はエラーにします。DEBUGでも入力全体の分離を確認します。

In [ ]:
image_sizes, split_counts = inspect_dataset()
print({'validated_images': split_counts})

## 10. YOLO形式への変換
person以外を除外し、vboxはextra.ignore、hboxはhead_attr.ignoreを独立に判定します。
標準YOLOはignore領域を表現できません。除外領域の検出は背景に対する誤検出として扱われる可能性があり、CrowdHuman公式評価とは異なります。
bboxをクリッピング・正規化し、不正データ件数だけ記録します。画像はシンボリックリンクで共用します。
DEBUGと本番で同じprocessedを使い回すことは拒否します。切替時は下の手順に従いprocessedを明示的に削除してください。

In [ ]:
"""Separate body/head ignore policy and streaming YOLO label conversion."""
import hashlib
import json
import math
import shutil
from collections import Counter
from pathlib import Path

NAMES = {'vbox': {0: 'person'}, 'hbox': {0: 'head'}, 'both': {0: 'person', 1: 'head'}}


def normalize_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        raise ValueError('Invalid bbox')
    if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in box):
        raise ValueError('Invalid bbox')
    x, y, w, h = box
    if width <= 0 or height <= 0 or w <= 0 or h <= 0:
        raise ValueError('Nonpositive dimensions')
    x1, y1 = max(0, min(width, x)), max(0, min(height, y))
    x2, y2 = max(0, min(width, x + w)), max(0, min(height, y + h))
    if x2 <= x1 or y2 <= y1:
        raise ValueError('Empty clipped bbox')
    return ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)


def ignored(annotation, attr):
    value = annotation.get(attr, {})
    if not isinstance(value, dict) or value.get('ignore', 0) not in (0, 1):
        raise ValueError('Invalid ignore attribute')
    return value.get('ignore', 0) == 1


def convert_boxes(boxes, mode, width, height):
    if mode not in NAMES:
        raise ValueError('Unsupported mode')
    labels, stats = [], Counter()
    fields = ('vbox', 'hbox') if mode == 'both' else (mode,)
    for annotation in boxes:
        if not isinstance(annotation, dict):
            stats['invalid_annotation'] += 1
            continue
        if annotation.get('tag') != 'person':
            stats['non_person'] += 1
            continue
        for field in fields:
            try:
                if ignored(annotation, 'extra' if field == 'vbox' else 'head_attr'):
                    stats[f'{field}_ignored'] += 1
                    continue
                box = annotation.get(field)
                normalized = normalize_box(box, width, height)
                class_id = 1 if mode == 'both' and field == 'hbox' else 0
                labels.append((class_id, *normalized))
                stats[f'{field}_accepted'] += 1
                x, y, w, h = box
                if x < 0 or y < 0 or x + w > width or y + h > height:
                    stats[f'{field}_clipped'] += 1
            except (ValueError, TypeError):
                stats[f'{field}_invalid'] += 1
    return labels, stats


def fingerprint(sizes, limit):
    digest = hashlib.sha256()
    for split in ('train', 'val'):
        with open(destination(f'annotation_{split}.odgt'), 'rb') as handle:
            for chunk in iter(lambda: handle.read(1024**2), b''):
                digest.update(chunk)
    digest.update(json.dumps(sorted(sizes.items())).encode())
    digest.update(str(limit).encode())
    return digest.hexdigest()


def convert_dataset(mode, sizes, limit=None):
    if mode not in NAMES or (limit is not None and limit <= 0):
        raise ValueError('Invalid conversion configuration')
    root = runtime_path(BASE / 'processed' / mode)
    signature = fingerprint(sizes, limit)
    manifest = root / 'conversion.json'
    if manifest.exists():
        previous = json.loads(manifest.read_text())
        if previous['fingerprint'] == signature:
            validate_labels(root, mode)
            return root / 'data.yaml'
        raise ValueError('Dataset changed; remove processed mode and matching run explicitly')
    staging = root.with_name(mode + '.building')
    if staging.exists():
        shutil.rmtree(runtime_path(staging))
    counts = Counter()
    selected = set()
    try:
        for split in ('train', 'val'):
            images = staging / 'images' / split
            labels_dir = staging / 'labels' / split
            images.mkdir(parents=True)
            labels_dir.mkdir(parents=True)
            for index, record in enumerate(read_odgt(destination(f'annotation_{split}.odgt'))):
                if limit is not None and index >= limit:
                    break
                key = record['ID']
                if key in selected:
                    raise ValueError('Duplicate image or split overlap')
                selected.add(key)
                width, height = sizes[key]
                source = runtime_path(BASE / 'raw/images' / f'{key}.jpg')
                if not source.is_file():
                    raise FileNotFoundError('Missing source image')
                rows, stats = convert_boxes(record['gtboxes'], mode, width, height)
                counts.update({f'{split}_{k}': v for k, v in stats.items()})
                counts[f'{split}_images'] += 1
                (images / source.name).symlink_to(source)
                (labels_dir / f'{key}.txt').write_text(''.join(' '.join([str(row[0]), *(f'{v:.10f}' for v in row[1:])]) + '\n' for row in rows))
        if not counts['train_images'] or not counts['val_images']:
            raise ValueError('Both splits must be nonempty')
        (staging / 'conversion.json').write_text(json.dumps({'fingerprint': signature, 'limit': limit, 'counts': counts}, indent=2))
        if root.exists():
            raise ValueError('Incomplete processed directory; remove it explicitly')
        staging.rename(root)
        write_yaml(root, mode)
        validate_labels(root, mode)
    except Exception:
        if staging.exists():
            shutil.rmtree(staging)
        raise
    return root / 'data.yaml'


def write_yaml(root, mode):
    import yaml
    root = runtime_path(root)
    data = {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': NAMES[mode]}
    (root / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False))


def validate_labels(root, mode):
    import yaml
    root = runtime_path(root)
    data = yaml.safe_load((root / 'data.yaml').read_text())
    if data != {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': NAMES[mode]}:
        raise ValueError('data.yaml mismatch')
    ids = []
    totals = {}
    for split in ('train', 'val'):
        images = list((root / 'images' / split).glob('*.jpg'))
        label_files = list((root / 'labels' / split).glob('*.txt'))
        if not images or {p.stem for p in images} != {p.stem for p in label_files}:
            raise ValueError('Images/labels mismatch')
        boxes = 0
        ids.append({p.stem for p in images})
        for image in images:
            runtime_path(image)
            if not image.is_file():
                raise FileNotFoundError('Broken image symlink')
        for path in label_files:
            runtime_path(path)
            for line in path.read_text().splitlines():
                fields = line.split()
                if len(fields) != 5 or fields[0] not in map(str, NAMES[mode]):
                    raise ValueError('Invalid class or label columns')
                values = list(map(float, fields[1:]))
                x, y, w, h = values
                if not all(math.isfinite(v) and 0 <= v <= 1 for v in values) or min(w, h) <= 0:
                    raise ValueError('Invalid normalized label')
                if x-w/2 < -1e-8 or x+w/2 > 1+1e-8 or y-h/2 < -1e-8 or y+h/2 > 1+1e-8:
                    raise ValueError('Label crosses image boundary')
                boxes += 1
        if not boxes:
            raise ValueError('Split has no usable labels')
        totals[split] = {'images': len(images), 'boxes': boxes}
    if ids[0] & ids[1]:
        raise ValueError('Split overlap')
    return totals

yaml_paths = {}
for mode in MODES_TO_RUN:
    yaml_paths[mode] = convert_dataset(mode, image_sizes, DEBUG_LIMIT if DEBUG_MODE else None)
    report = __import__('json').loads((yaml_paths[mode].parent / 'conversion.json').read_text())
    print(mode, report['counts'])

## 11. 合成テストとアノテーションの検証
合成画像・合成ODGTで変換処理をテストします。実画像の表示は行いません。
クラスID・正規化座標・画像とラベルの対応・split分離を検証します。空ラベル画像は背景画像として残します。

In [ ]:
# この合成テストもノートブック内で直接実行します。実データは使いません。
import io
import tempfile
import unittest
from unittest.mock import patch
from PIL import Image


class NotebookSyntheticTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory(prefix='synthetic-notebook-')
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)
        # 合成データに限り、一時ディレクトリを使って保存先を差し替えます。
        self.guard = patch.dict(globals(), BASE=self.root)
        self.guard.start()
        self.addCleanup(self.guard.stop)
        for relative in ('raw/annotations', 'raw/images', 'processed'):
            (self.root / relative).mkdir(parents=True)
        self.person = {'tag': 'person', 'vbox': [10, 20, 30, 40], 'hbox': [20, 20, 10, 10]}
        for split in ('train', 'val'):
            key = f'synthetic_{split}'
            Image.new('RGB', (100, 100)).save(self.root / 'raw/images' / f'{key}.jpg')
            record = {'ID': key, 'gtboxes': [self.person]}
            destination(f'annotation_{split}.odgt').write_text(json.dumps(record) + '\n')

    def test_vbox(self):
        rows, _ = convert_boxes([self.person], 'vbox', 100, 100)
        self.assertEqual(rows, [(0, .25, .4, .3, .4)])

    def test_hbox(self):
        rows, _ = convert_boxes([self.person], 'hbox', 100, 100)
        self.assertEqual(rows, [(0, .25, .25, .1, .1)])

    def test_independent_ignore(self):
        for body, head, expected in ((0, 0, [0, 1]), (1, 0, [1]), (0, 1, [0]), (1, 1, [])):
            annotation = dict(self.person, extra={'ignore': body}, head_attr={'ignore': head})
            rows, _ = convert_boxes([annotation], 'both', 100, 100)
            self.assertEqual([row[0] for row in rows], expected)

    def test_clipping(self):
        self.assertEqual(normalize_box([-20, -10, 150, 120], 100, 100), (.5, .5, 1, 1))

    def test_invalid_boxes(self):
        for box in (None, [], [0, 0, 0, 1], [0, 0, -1, 1], [101, 0, 5, 5], [0, 0, float('nan'), 1]):
            rows, stats = convert_boxes([dict(self.person, vbox=box)], 'vbox', 100, 100)
            self.assertEqual(rows, [])
            self.assertEqual(stats['vbox_invalid'], 1)

    def test_non_person(self):
        rows, _ = convert_boxes([dict(self.person, tag='mask')], 'both', 100, 100)
        self.assertEqual(rows, [])

    def test_class_ids(self):
        rows, _ = convert_boxes([self.person], 'both', 100, 100)
        self.assertEqual([row[0] for row in rows], [0, 1])
        self.assertEqual(NAMES['both'], {0: 'person', 1: 'head'})

    def test_split_separation(self):
        sizes, counts = inspect_dataset()
        root = convert_dataset('vbox', sizes).parent
        self.assertEqual(counts, {'train': 1, 'val': 1})
        self.assertTrue((root / 'images/train/synthetic_train.jpg').is_symlink())
        self.assertFalse((root / 'images/val/synthetic_train.jpg').exists())
        self.assertTrue((root / 'labels/val/synthetic_val.txt').exists())

    def test_yaml(self):
        import yaml
        sizes, _ = inspect_dataset()
        for mode in ('vbox', 'hbox', 'both'):
            path = convert_dataset(mode, sizes)
            data = yaml.safe_load(path.read_text())
            self.assertEqual(data['path'], str(path.parent))
            self.assertEqual(data['train'], 'images/train')
            self.assertEqual(data['val'], 'images/val')
            self.assertEqual(data['names'], NAMES[mode])
            self.assertEqual(validate_labels(path.parent, mode)['train']['images'], 1)

    def test_ignore_rules(self):
        with tempfile.TemporaryDirectory(prefix='synthetic-git-policy-') as directory:
            root = Path(directory)
            (root / '.gitignore').write_text(GITIGNORE_CONTENT)
            subprocess.run(['git', 'init', '-q', str(root)], check=True)
            for name in ('CrowdHuman/a', 'crowdhuman/a', 'datasets/a', 'data/a', 'a.odgt', 'a.zip',
                         'a.jpg', 'a.jpeg', 'a.png', 'a.webp', 'runs/a', 'weights/a', 'checkpoints/a',
                         'a.pt', 'a.pth', 'a.onnx', '.env', '.env.local', 'a.key', '__pycache__/a', '.ipynb_checkpoints/a'):
                self.assertEqual(subprocess.run(['git', '-C', str(root), 'check-ignore', '-q', '--', name]).returncode, 0)

    def test_missing_file(self):
        with self.assertRaises(FileNotFoundError):
            list(read_odgt(self.root / 'raw/annotations/missing.odgt'))
        with self.assertRaises(ValueError):
            image_size(self.root / 'raw/images/missing.jpg')

    def test_storage_boundary(self):
        with self.assertRaises(ValueError):
            runtime_path(self.root.parent / 'outside')


suite = unittest.defaultTestLoader.loadTestsFromTestCase(NotebookSyntheticTests)
with local_console(BASE / 'synthetic_tests.log'):
    result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise RuntimeError('合成テスト失敗。synthetic_tests.logを確認してください。')
print({'synthetic_tests': result.testsRun, 'status': 'passed'})
for mode in MODES_TO_RUN:
    print(mode, validate_labels(yaml_paths[mode].parent, mode))

## 12. data.yamlの確認
変換時にdata.yamlを生成済みです。各モードにimages/train・images/val・labels/train・labels/valがあります。
vboxは0:person、hboxは0:head、bothは0:person / 1:headです。データの内容は表示しません。

In [ ]:
for mode in MODES_TO_RUN:
    report = validate_labels(yaml_paths[mode].parent, mode)
    print({'mode': mode, 'data_yaml_valid': True, 'counts': report})

## 13. YOLOモデルの読み込み
参考実装のyolo26n.pt（設定でyolo26s.ptも選択可能）を使用します。
公式Ultralytics重みをランタイムに取得し、同じセッションではキャッシュを再利用します。
外部からアップロードした信頼できない.ptは読み込みません。

In [ ]:
"""Local-only, reproducible training with explicit checkpoint resumption."""
import importlib.metadata
import json
import os
import random
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainConfig:
    mode: str = 'vbox'
    model: str = 'yolo26n.pt'
    epochs: int = 50
    batch: int = 16
    imgsz: int = 640
    seed: int = 42
    patience: int = 10
    workers: int = 2
    debug: bool = False
    run_name: str = 'experiment'
    allow_cpu: bool = False

    def validate(self):
        if self.mode not in NAMES or min(self.epochs, self.batch, self.imgsz) <= 0:
            raise ValueError('Invalid training configuration')
        if self.imgsz % 32 or self.workers < 0 or self.patience < 0:
            raise ValueError('Invalid image size, workers or patience')
        if self.model not in ('yolo26n.pt', 'yolo26s.pt'):
            raise ValueError('Use the reference YOLO26n/s model')
        if not self.run_name or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in self.run_name):
            raise ValueError('Unsafe run name')

    @property
    def run_dir(self):
        self.validate()
        return runtime_path(BASE / 'runs' / self.mode / (self.run_name + ('_debug' if self.debug else '')))


def versions():
    names = ('ultralytics', 'torch', 'torchvision', 'Pillow', 'PyYAML', 'onnx', 'onnxruntime')
    result = {}
    for name in names:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = 'not installed'
    return result


def device_for(config):
    import torch
    if torch.cuda.is_available():
        return 0
    warnings.warn('GPUがありません。ColabのGPUランタイムを選択してください。', stacklevel=2)
    if not config.allow_cpu:
        raise RuntimeError('GPU required; set allow_cpu=True only for a deliberate CPU test')
    return 'cpu'


def load_model(config):
    config.validate()
    YOLO = private_yolo()
    previous = Path.cwd()
    try:
        os.chdir(runtime_path(BASE / 'weights'))
        return YOLO(str(BASE / 'weights' / config.model))
    finally:
        os.chdir(previous)


def train(config, resume=False):
    config.validate()
    import numpy as np
    import torch
    device = device_for(config)
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    root = runtime_path(BASE / 'processed' / config.mode)
    validate_labels(root, config.mode)
    conversion = json.loads((root / 'conversion.json').read_text())
    if (conversion['limit'] is not None) != config.debug:
        raise ValueError('DEBUG flag and converted data disagree')
    run = config.run_dir
    meta_path = run / 'experiment.json'
    specification = {'config': asdict(config), 'dataset_fingerprint': conversion['fingerprint'], 'versions': versions()}
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        if not resume:
            raise FileExistsError('Run exists; explicitly resume or select a new run_name')
        if meta['specification'] != specification:
            raise ValueError('Checkpoint configuration/dataset/library mismatch')
    else:
        if resume or run.exists():
            raise ValueError('No matching resumable experiment metadata')
        run.mkdir(parents=True)
        meta = {'specification': specification, 'sessions': []}
        meta_path.write_text(json.dumps(meta, indent=2))
    checkpoint = runtime_path(run / 'weights/last.pt')
    YOLO = private_yolo()
    if resume:
        if not checkpoint.is_file():
            raise FileNotFoundError('Missing last.pt checkpoint')
        # Only our trusted runtime checkpoint; never load arbitrary uploaded pickle files.
        ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
        if ckpt.get('epoch', -1) < 0 or ckpt.get('optimizer') is None:
            raise ValueError('Completed/stripped checkpoint cannot resume; start a new run')
        del ckpt
    started = time.perf_counter()
    status = 'interrupted'
    try:
        # Keep epoch progress and diagnostic output out of the notebook document.
        with local_console(run / 'console.log'):
            model = YOLO(str(checkpoint)) if resume else load_model(config)
            if resume:
                model.train(resume=True, plots=False, save_json=False, save_txt=False, device=device)
            else:
                model.train(data=str(root / 'data.yaml'), epochs=config.epochs, batch=config.batch,
                            imgsz=config.imgsz, seed=config.seed, deterministic=True,
                            device=device, project=str(run.parent), name=run.name, exist_ok=True,
                            patience=config.patience, workers=config.workers, cache=False,
                            single_cls=False, cos_lr=True, close_mosaic=min(10, config.epochs),
                            shear=0.0, perspective=0.0, amp=True, plots=False,
                            max_det=1000, save=True, save_period=-1,
                            save_json=False, save_txt=False, verbose=False)
        status = 'completed'
    finally:
        meta['sessions'].append({'seconds': time.perf_counter()-started, 'status': status, 'resume': resume})
        meta_path.write_text(json.dumps(meta, indent=2))
    return run

configs = [TrainConfig(mode=mode, model=BASE_MODEL, epochs=EPOCHS, batch=BATCH_SIZE,
                       imgsz=IMAGE_SIZE, seed=SEED, debug=DEBUG_MODE, run_name=RUN_NAME,
                       allow_cpu=ALLOW_CPU) for mode in MODES_TO_RUN]
for config in configs:
    config.validate()
if not RESUME:
    with local_console(BASE / 'model_setup.log'):
        base_model = load_model(configs[0])
        del base_model
print('モデル準備完了')

## 14. 追加学習とチェックポイント管理
同じ初期モデル・seed・学習条件で各モードを学習します。結果はruns/<mode>/<RUN_NAME>です。
進捗や画像はノートブックに表示せず、ログはランタイムに保存します。T4でOOMが出たらbatchを8などに下げ、新しい実験名を使ってください。
中断時のlast.ptから同じセッション内で再開できます。完了後にoptimizerが除去された重みはresumeできません。
本番比較はearly stoppingで実際のepoch数が異なる可能性もあるため、training CSVも確認してください。

In [ ]:
trained_runs = {}
for config in configs:
    trained_runs[config.mode] = train(config, resume=RESUME)
    print(config.mode, '学習完了')

## 15. 評価
各モデルをそれぞれのval正解ラベルで評価します。Precision・Recall・mAP50・mAP50-95、学習時間、推論ms/画像、重みMiBを記録します。
速度はbatch=1・FP32でのUltralytics validationのinference値です。前後処理、ブラウザ、追跡器を含むアプリ全体の速度ではありません。
学習時間は再開を含むセッション時間で、準備処理も含みます。

In [ ]:
"""Scalar-only evaluation, comparable CSVs and explicitly selected local exports."""
from dataclasses import asdict
import csv
import hashlib
import json
import shutil
from pathlib import Path


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def evaluate(config):
    run = config.run_dir
    best = runtime_path(run / 'weights/best.pt')
    if not best.is_file():
        raise FileNotFoundError('No trained best.pt')
    experiment = json.loads((run / 'experiment.json').read_text())
    if experiment['specification']['config'] != asdict(config):
        raise ValueError('Evaluation must use the saved training configuration')
    root = runtime_path(BASE / 'processed' / config.mode)
    validate_labels(root, config.mode)
    conversion = json.loads((root / 'conversion.json').read_text())
    if conversion['fingerprint'] != experiment['specification']['dataset_fingerprint']:
        raise ValueError('Evaluation dataset differs from the training dataset')
    YOLO = private_yolo()
    with local_console(run / 'evaluation.log'):
        model = YOLO(str(best))
        metrics = model.val(data=str(BASE / 'processed' / config.mode / 'data.yaml'),
                            split='val', imgsz=config.imgsz, batch=1, device=device_for(config),
                            workers=config.workers, seed=config.seed, conf=0.001, iou=0.7,
                            max_det=1000, half=False, plots=False, save=False,
                            save_json=False, save_txt=False, verbose=False,
                            project=str(run), name='validation', exist_ok=True)
    import torch
    identity = {'mode': config.mode, 'run_name': run.name, 'weight_sha256': sha256(best)}
    common = dict(precision=float(metrics.box.mp), recall=float(metrics.box.mr),
                  map50=float(metrics.box.map50), map50_95=float(metrics.box.map),
                  training_seconds=sum(s['seconds'] for s in experiment['sessions']),
                  inference_ms_per_image=float(metrics.speed['inference']),
                  model_mib=best.stat().st_size/1024**2,
                  device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
    record = {**identity, **common, 'specification': experiment['specification'],
              'evaluation': {'batch': 1, 'imgsz': config.imgsz, 'half': False, 'conf': 0.001, 'iou': 0.7, 'max_det': 1000},
              'classes': []}
    for index, class_id in enumerate(metrics.box.ap_class_index):
        p, r, ap50, ap = metrics.box.class_result(index)
        record['classes'].append({'class_id': int(class_id), 'name': model.names[int(class_id)],
                                  'precision': float(p), 'recall': float(r),
                                  'map50': float(ap50), 'map50_95': float(ap)})
    (run / 'evaluation.json').write_text(json.dumps(record, indent=2))
    return {**identity, **common}


def comparison(output=BASE / 'exports/comparison.csv'):
    rows, signatures = [], []
    for path in sorted((BASE / 'runs').glob('*/*/evaluation.json')):
        runtime_path(path)
        row = json.loads(path.read_text())
        best = runtime_path(path.parent / 'weights/best.pt')
        if not best.is_file() or sha256(best) != row['weight_sha256']:
            raise ValueError('Stale evaluation: evaluate the current checkpoint again')
        spec = json.loads(json.dumps(row['specification']))
        for key in ('mode', 'run_name'):
            spec['config'].pop(key)
        signatures.append(json.dumps({'spec': spec, 'device': row['device'], 'evaluation': row['evaluation']}, sort_keys=True))
        scalar = {k: v for k, v in row.items() if k not in ('specification', 'evaluation', 'classes')}
        rows.append({**scalar, 'scope': 'all', 'class_name': 'all'})
        for cls in row['classes']:
            rows.append({**scalar, **{k: cls[k] for k in ('precision', 'recall', 'map50', 'map50_95')},
                         'scope': 'class', 'class_name': cls['name']})
    if not rows:
        raise FileNotFoundError('No evaluation results; train and evaluate at least one mode')
    comparable = len(set(signatures)) == 1
    for row in rows:
        row['matched_conditions'] = comparable
    output = runtime_path(output)
    with open(output, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return rows


def inspect_onnx(path, image_size, contract=None):
    """Run synthetic input only. A supplied application contract is required for matching."""
    import numpy as np
    import onnx
    import onnxruntime as ort
    path = runtime_path(path)
    onnx.checker.check_model(onnx.load(str(path)))
    session = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inputs = session.get_inputs()
    if len(inputs) != 1 or inputs[0].type != 'tensor(float)' or len(inputs[0].shape) != 4:
        raise ValueError('Unexpected ONNX image input')
    tensor = np.zeros((1, 3, image_size, image_size), dtype=np.float32)
    outputs = session.run(None, {inputs[0].name: tensor})
    if not outputs or any(not np.isfinite(value).all() for value in outputs):
        raise ValueError('Invalid ONNX synthetic inference output')
    interface = {'input': {'name': inputs[0].name, 'dtype': inputs[0].type, 'shape': list(tensor.shape)},
                 'outputs': [{'name': meta.name, 'dtype': meta.type, 'shape': list(value.shape)}
                             for meta, value in zip(session.get_outputs(), outputs)]}
    if contract is not None and interface != contract:
        raise ValueError('Web application input/output contract mismatch')
    return {'interface': interface, 'onnx_checker': 'passed', 'synthetic_inference': 'passed',
            'application_contract': 'matched' if contract is not None else 'unverified',
            'browser_integration': 'unverified',
            'preprocessing': 'RGB float32 NCHW /255; letterbox and inverse coordinate mapping required',
            'postprocessing': 'Confirm YOLO26 end-to-end output vs raw detections before connecting tracker'}


def export_model(config, onnx_export=True, contract=None):
    YOLO = private_yolo()
    best = runtime_path(config.run_dir / 'weights/best.pt')
    if not best.is_file():
        raise FileNotFoundError('No trained best.pt')
    prefix = f'{config.mode}_{config.run_dir.name}'
    out = BASE / 'exports'
    weight = runtime_path(out / f'{prefix}.pt')
    shutil.copyfile(best, weight)
    csv_source = runtime_path(config.run_dir / 'results.csv')
    if csv_source.is_file():
        shutil.copyfile(csv_source, out / f'{prefix}_training.csv')
    if onnx_export:
        with local_console(config.run_dir / 'export.log'):
            model = YOLO(str(weight))
            exported = runtime_path(model.export(format='onnx', imgsz=config.imgsz, opset=13,
                                                 dynamic=True, simplify=False, nms=False,
                                                 half=False, batch=1, device='cpu'))
            report = inspect_onnx(exported, config.imgsz, contract)
        report['names'] = model.names
        report['weight_sha256'] = sha256(weight)
        (out / f'{prefix}_onnx_interface.json').write_text(json.dumps(report, indent=2))
    # Deliberately no zip of the run directory or image/annotation inclusion.
    return [p.name for p in sorted(out.glob(prefix + '*')) if p.suffix in ('.pt', '.onnx', '.csv')]


def download_selected(names):
    """Called only from an opt-in notebook cell; no external publication."""
    from google.colab import files
    for name in names:
        if Path(name).name != name or Path(name).suffix not in ('.pt', '.onnx', '.csv'):
            raise ValueError('Only explicitly selected pt/onnx/csv exports may be downloaded')
        path = runtime_path(BASE / 'exports' / name)
        if not path.is_file():
            raise FileNotFoundError('Selected export does not exist')
        files.download(str(path))

for config in configs:
    print(evaluate(config))

## 16. vbox・hboxの比較
検出対象が異なるためmAPだけで優劣は判定できません。bothではクラス別の行も出力します。
複数モードを実行するか、設定のモードを切り替えて学習・評価を追加後、このセルを再実行してください。
matched_conditionsは保存した設定・データ指紋・依存・評価条件・GPU名の一致を表します。対象の同一性や完全な再現性を保証する値ではありません。
重みが変わった後の古い評価結果はエラーになります。

In [ ]:
comparison_rows = comparison()
print({'comparison_csv': f'{EXPORT_DIR}/comparison.csv', 'rows': len(comparison_rows),
       'matched_conditions': all(row['matched_conditions'] for row in comparison_rows)})
for row in comparison_rows:
    if row['scope'] == 'all':
        print(row)

## 17. モデルのエクスポート
exportsに.pt・ONNX・学習CSVを出力します。画像・ラベル・run一式のZIPは作成しません。
ONNXはopset 13・dynamic・FP32、simplify=Falseで出力し、checkerと合成入力によるONNX Runtime CPU推論を実行します。
入出力名・shape・dtype・クラス定義は*_onnx_interface.jsonに記録します。YOLO26のend-to-end出力は、従来のYOLOの生出力を想定するアプリと異なる場合があります。
WEBAPP_CONTRACTに実アプリの入出力辞書を設定すると完全一致を検査します。未指定は互換性未検証です。
RGB/NCHW/0〜1、letterbox・座標逆変換、confidence・クラスID・NMS有無、ONNX Runtime Web対応opsetをアプリ側で検証してください。

In [ ]:
exported_names = []
for config in configs:
    exported_names.extend(export_model(config, onnx_export=EXPORT_ONNX, contract=WEBAPP_CONTRACT))
print({'exported': exported_names})

## 18. 必要ファイルの明示ダウンロード
このセルは初期状態ではダウンロードしません。必要なファイル名をDOWNLOAD_NAMESに列挙し、DO_DOWNLOAD=Trueに変更して、このセルだけを明示的に実行してください。
.pt・.onnx・.csvに限定します。公開操作ではありません。学習済みモデルにもデータ由来の情報が含まれる可能性があるため、外部公開・商用利用前に条件を確認してください。

In [ ]:
DO_DOWNLOAD = False
DOWNLOAD_NAMES = []  # 例: ['vbox_experiment.pt', 'comparison.csv']
if DO_DOWNLOAD:
    download_selected(DOWNLOAD_NAMES)
else:
    print('ダウンロード未実行')

## 19. 一時データの削除（任意）
学習済みモデルの取得を確認してから実行してください。初期状態では削除しません。
DELETE_DATA=Trueはrawとprocessedを削除します。DELETE_RUNS=Trueはチェックポイントと学習ログも削除し、再開できなくなります。
DEBUGから本番へ切り替える際は、DELETE_DATAを使うとrawも消えます。rawを再利用する場合はColabファイル欄でprocessed/<mode>だけを明示的に削除し、新しいRUN_NAMEで変換・学習してください。
exportsは残ります。ランタイムのリセットではexportsも含めて消失します。

In [ ]:
DELETE_DATA = False
DELETE_RUNS = False
import shutil
if DELETE_DATA:
    for part in ('raw', 'processed'):
        target = runtime_path(BASE / part)
        if target.exists():
            shutil.rmtree(target)
if DELETE_RUNS:
    target = runtime_path(BASE / 'runs')
    if target.exists():
        shutil.rmtree(target)
print('削除設定に従って処理しました')